# Ex.No 1 — Lexical Analyzer to Recognize Patterns in C (with Symbol Table)


## AIM
To develop a lexical analyzer using FLEX to recognize tokens such as identifiers, constants, comments, and operators in a C program, and to create a symbol table while recognizing identifiers.


## ALGORITHM / PROCEDURE
1. Start the program by including the necessary headers within the FLEX definitions section (`%{ ... %}`).
2. Define regular expressions for:
   - Identifiers: `[a-zA-Z_][a-zA-Z0-9_]*`
   - Constants: `[0-9]+`
   - Comments: `//.*` and `/* ... */`
   - Operators: `+ - * / = < >`
3. Declare a symbol table array (structure with `name` and `type` fields) in the definitions section.
4. Write rules in the rules section of the FLEX (`.l`) file:
   - When an identifier is recognized, call `insert()` to add it to the symbol table if not already present.
   - Print / categorize constants, operators, and comments as they are matched.
5. Use `yylex()` actions to print matched tokens and perform symbol table insertion.
6. In `main()`, open the input file, call `yylex()`, then print the symbol table.
7. Compile the FLEX file using `flex` and `gcc`. Execute the program with a sample C code input file.
8. Stop.

**Procedure**
1. Create a new FLEX file, `symtab.l`.
2. In the definitions section, include headers and declare the symbol table array along with `insert()`/`lookup()` helper functions.
3. In the rules section, define patterns for identifiers, constants, comments, and operators.
4. Use `{ printf(...) }` actions to print the recognized tokens and call `insert()` for identifiers.
5. Compile: `flex symtab.l` then `gcc lex.yy.c -o symtab -lfl`.
6. Run: `./symtab input.c` and observe the output / symbol table entries.


## PSEUDOCODE / LOGIC
```
BEGIN
    DECLARE symbol_table[]                     // array of {name, type}

    FUNCTION lookup(name):
        FOR each entry in symbol_table:
            IF entry.name == name THEN RETURN index
        RETURN -1

    FUNCTION insert(name):
        IF lookup(name) == -1 THEN
            APPEND {name, type=1} to symbol_table

    OPEN input file
    FOR each lexeme scanned from input:
        IF lexeme matches comment pattern      -> PRINT "Comment : " + lexeme
        ELSE IF lexeme matches identifier      -> insert(lexeme); PRINT "Identifier : " + lexeme
        ELSE IF lexeme matches digit sequence  -> PRINT "Constant : " + lexeme
        ELSE IF lexeme matches operator symbol -> PRINT "Operator : " + lexeme
        ELSE IF lexeme is whitespace           -> SKIP
    PRINT symbol_table entries (S.No, Name)
END
```


## PROGRAM & OUTPUT
The cells below contain the source program (FLEX/BISON/C) and its executed output.


In [ ]:
!apt-get update -qq
!apt-get install -y flex gcc

In [3]:
%%writefile symtab.l
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

struct symtab {
    char name[30];
    int type;
} symtab[100];

int sc = 0;

int lookup(char *s) {
    int i;
    for (i = 0; i < sc; i++)
        if (strcmp(symtab[i].name, s) == 0)
            return i;
    return -1;
}

void insert(char *s) {
    if (lookup(s) == -1) {
        strcpy(symtab[sc].name, s);
        symtab[sc].type = 1;
        sc++;
    }
}
%}

DIGIT [0-9]
ID [a-zA-Z_][a-zA-Z0-9_]*

%%

"/*"([^*]|\*+[^*/])*\*+"/" {
    printf("Comment : %s\n", yytext);
}

"//".* {
    printf("Comment : %s\n", yytext);
}

{ID} {
    insert(yytext);
    printf("Identifier : %s\n", yytext);
}

{DIGIT}+ {
    printf("Constant : %s\n", yytext);
}

"+"|"-"|"*"|"/"|"="|"<"|">" {
    printf("Operator : %s\n", yytext);
}

[ \t\n] {
    /* skip whitespace */
}

. {
    /* ignore other characters */
}

%%

int yywrap() {
    return 1;
}

int main(int argc, char *argv[]) {

    if (argc < 2) {
        printf("Usage: %s <input file>\n", argv[0]);
        return 1;
    }

    yyin = fopen(argv[1], "r");

    if (!yyin) {
        printf("Cannot open file %s\n", argv[1]);
        return 1;
    }

    yylex();

    printf("\nSYMBOL TABLE\n");
    printf("S.No\tName\n");

    int i;
    for (i = 0; i < sc; i++)
        printf("%d\t%s\n", i + 1, symtab[i].name);

    fclose(yyin);

    return 0;
}

Overwriting symtab.l


In [6]:
!flex symtab.l
!gcc lex.yy.c -o symtab

In [9]:
%%writefile input.txt
int a = 10;
float b = 20;
a = a + b;
// this is a comment

Overwriting input.txt


In [8]:
!./symtab input.txt

Identifier : int
Identifier : a
Operator : =
Constant : 10
Identifier : float
Identifier : b
Operator : =
Constant : 20
Identifier : a
Operator : =
Identifier : a
Operator : +
Identifier : b
Comment : // this is a comment

SYMBOL TABLE
S.No	Name
1	int
2	a
3	float
4	b


## RESULT
Thus the FLEX program to develop a lexical analyzer recognizing identifiers, constants, comments and operators, and to build a symbol table, was executed and verified successfully.
